퍼셉트론
 forward : x -> z1 = w1 * x + b1  -> a1 = h(z1) -> z2 = w2 * a1 + b2 -> a2 = sig(z2)
 loss : L = loss(a2) 
 backard : dL/dw2 = dL/da2 * da2/dz2 * dz2/dw2
           dL/dw1 = dL/da2 * da2/dz2 * dz2/da1 * da1/dz1 * dz1/dw1
1. 클래스 생성(class NeuralNetwork)
2. 생성자(인풋수, 히든수, 아웃풋수)
    인풋 수의 의미는 특성의 수(차원) *배치 수는 특성과 관련이 없다.
    히든 수의 의미는 특성에 가중치를 거는 방법의 수(특성 차원을 늘릴 수도, 줄일 수도 있음)
    아웃풋 수의 의미는 결과가 가질 수 있는 경우의 수(회귀:1, 이진분류:2, 여러개 중 선택:여러개)

    예를 들어 X(m,n)이 입력된다고 하면, 배치 수:m, 특성차원:n
    이에 따라 가중치는 n개의 행을 가지며, 히든 노드 수만큼의 컬럼을 가진다.
    즉 weight는 W(n, h)
    편향(b)는 출력 단의 수만큼이므로 bias = b(1, h)

    z = X @ W + b -> (m, n) @ (n, h) + (1, h):브로드캐스팅
    z는 배치 수만큼의 행을 가진 (m,1) 벡터를 열로 하는 (m, h) 행렬
    
2-2. bias(input->hidden)
3. 활성화 함수, 미분
4. 순전파
5. 손실함수
6. 역전파
7. 훈련(epoch, learning_rate)

In [ ]:
import numpy as np 

class ANN:
    def __init__(self, input_size=2, hidden_size=4, output_size=1):
        # 가중치(입력 X : (m, n))
        self.W1 = np.random.randn(input_size, hidden_size)  # (n, h)
        self.b1 = np.zeros((1, hidden_size))                  # (1, h)
        self.W2 = np.random.randn(hidden_size, output_size) # (h, o)
        self.b2 = np.zeros((1, output_size))                  # (1, o)
    
    def relu(self, x):
        return np.maximum(x, 0)

    def relu_derivative(self, x):
        return np.where(x > 0, 1, 0)

    def sigmoid(self, x):
        # x (m, n) -> (m, n)
        return 1 / (1 + np.exp(-x))
    
    def sigmoid_derivative(self, x):
        y = self.sigmoid(x)
        return y * (1 - y)

    def forward(self, X):
        self.X = X
        self.z1 = self.X @ self.W1 + self.b1          # (m, h)
        self.a1 = self.relu(self.z1)     # (m, h)
        self.z2 = self.a1 @ self.W2 + self.b2    # (m, o)
        self.a2 = self.sigmoid(self.z2)     # (m, o)
        return self.a2

    def loss(self, Y):
        # Y : (m, o)
        return np.mean((Y - self.a2) ** 2)   # (m, o)

    def backward(self, Y):
        # back propagation to hidden layer
        # dL/dw2 = dL/da2 * da2/dz2 * dz2/dw
        #        = delta2           * a1
        dL_da2 = 2 * (Y - self.a2) * -1         # (m, o)
        da2_dz2 = self.sigmoid_derivative(self.a2)   # (m, o)
        delta2 = dL_da2 * da2_dz2               # (m, o) - elementwise
        self.dL_dw2 = self.a1.T @ delta2             # (h, m @ (m, o) = (h, o)
        self.dL_db2 = np.sum(delta2, axis=0, keepdims=True) # (1, o)

        # back propagation to input layer
        # dL/dw1 = dL/da2 * da2/dz2 * dz2/da1 * da1/dz1 * dz1/dw1
        # delta1 = delta2   (m, o)
        #          * w2     (h, o)
        #          * sig_dev(a1) (m, n)
        # dL/dw1 = delta1 * X      (m, n)
        delta1 = (delta2 @ self.W2.T # (m, o) @ (o, h) = (m, h)
                    * self.relu_derivative(self.z1))  # (m, h) 
        self.dL_dw1 = self.X.T @ delta1   # (n, m) @ (m, h) = (n, h)
        self.dL_db1 = np.sum(delta1, axis=0, keepdims=True) # (1, h)

    def optimize(self, learning_rate):
        self.W1 -= learning_rate * self.dL_dw1
        self.W2 -= learning_rate * self.dL_dw2
        self.b1 -= learning_rate * self.dL_db1
        self.b2 -= learning_rate * self.dL_db2

    def fit(self, X, Y, learning_rate=0.01, epoch=1000):
        print('transpose of input matrix')
        print(X.T)
        print('\ntranspose of label matrix')
        print(Y.T)
        losses = []
        every_100 = 0
        for i in range(epoch):
            a = self.forward(X)
            losses.append(self.loss(Y))
            self.backward(Y)
            self.optimize(learning_rate)
            pred = (a > 0.5).astype(int)
            if (pred == Y).all():
                print(f'\npredicted after {i} epochs')
                print('predicted value')
                # print(np.column_stack((pred, Y)))
                print(pred.T)
                print('weight matrix_1', self.W1, sep='\n')
                print('weight matrix_2', self.W2, sep='\n')
                return
            if i == every_100:
                print('Epoch :', i, '\tLoss :', self.loss(Y))
                print('Prediction :', pred.T, sep='\n')
                every_100 += 100

    def predict(self, X):
        a = self.forward(X)
        pred = (a > 0.5).astype(int)


In [191]:
model = ANN()
X = np.random.randint(0, 2, size=(30, 2))
Y = (np.sum(X, axis=1) == 1).astype(int).reshape(-1, 1) #XOR
Y2 = (np.sum(X, axis=1) > 0).astype(int).reshape(-1, 1)  #OR

model.fit(X, Y)

transpose of input matrix
[[0 0 1 1 0 1 1 1 0 1 1 1 1 1 1 1 1 0 1 0 0 1 1 0 1 1 0 0 0 0]
 [1 1 1 1 1 0 1 0 0 1 0 1 0 1 1 1 1 1 0 0 1 1 0 0 0 0 1 0 0 0]]

transpose of label matrix
[[1 1 0 0 1 1 0 1 0 0 1 0 1 0 0 0 0 1 1 0 1 0 1 0 1 1 1 0 0 0]]
Epoch : 0 	Loss : 0.2867702721118119
Prediction :
[[0 0 1 1 0 1 1 1 0 1 1 1 1 1 1 1 1 0 1 0 0 1 1 0 1 1 0 0 0 0]]

predicted after 85 epochs
predicted value
[[1 1 0 0 1 1 0 1 0 0 1 0 1 0 0 0 0 1 1 0 1 0 1 0 1 1 1 0 0 0]]
weight matrix_1
[[ 2.00271747  1.52824239  0.83155747  1.0487017 ]
 [-1.99614582 -0.24816799 -0.22197324 -0.34240322]]
weight matrix_2
[[ 2.07917677]
 [ 0.32750963]
 [-1.12781632]
 [-1.13486122]]


In [194]:
pred = model.forward(X)

In [200]:
(pred > 0.5).astype(int).T

array([[1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0,
        1, 0, 1, 1, 1, 0, 0, 0]])